# 03 — Directional Probability Model
LSTM-MDN with Student-t mixture, XGBoost prior, deep ensemble, Venn-ABERS calibration.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import stats
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
import xgboost as xgb
import matplotlib.pyplot as plt
from pathlib import Path
from config import *
from utils import compute_strike_probability

DATA_DIR = Path('../data')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
features = pd.read_parquet(DATA_DIR / 'features.parquet')
regimes = pd.read_parquet(DATA_DIR / 'regimes.parquet')

# Merge regime posteriors into features
regime_cols = [c for c in regimes.columns if c.startswith('hmm_prob_') or c == 'regime_score']
features = features.join(regimes[regime_cols], how='left')

# Define feature columns (exclude price, target, metadata)
exclude = ['spot_close', 'perp_close', 'target_up', 'log_return_1m']
feat_cols = [c for c in features.columns if c not in exclude]
target_col = 'target_up'

# Drop rows with NaN target
features = features.dropna(subset=[target_col])
print(f'Features: {len(feat_cols)}  |  Samples: {len(features):,}')

## XGBoost Prior

In [ ]:
# Train XGBoost on tabular features to extract prior signals
X_tab = features[feat_cols].fillna(0).values
y_tab = features[target_col].values

# Simple time-based split for XGBoost
split_idx = int(len(X_tab) * 0.8)
X_train_xgb, X_val_xgb = X_tab[:split_idx], X_tab[split_idx:]
y_train_xgb, y_val_xgb = y_tab[:split_idx], y_tab[split_idx:]

# Binary classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train_xgb, y_train_xgb,
              eval_set=[(X_val_xgb, y_val_xgb)], verbose=False)

# Quantile regression for uncertainty
xgb_q10 = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    objective='reg:quantileerror', quantile_alpha=0.1, random_state=42
)
xgb_q90 = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    objective='reg:quantileerror', quantile_alpha=0.9, random_state=42
)
xgb_q10.fit(X_train_xgb, y_train_xgb, verbose=False)
xgb_q90.fit(X_train_xgb, y_train_xgb, verbose=False)

# Extract 3 prior features
xgb_prob = xgb_model.predict_proba(X_tab)[:, 1]
xgb_uncertainty = xgb_q90.predict(X_tab) - xgb_q10.predict(X_tab)

# BS implied probability with drift
rv_col = 'rv_5m' if 'rv_5m' in features.columns else feat_cols[0]
sigma_hat = np.sqrt(features[rv_col].fillna(0).values.clip(0))
tau = 15 / 1440  # 15 min as fraction of day
mu_hat = features['log_return_1m'].fillna(0).rolling(15).mean().fillna(0).values
d2 = np.where(sigma_hat > 0,
              (mu_hat - 0.5 * sigma_hat**2) * tau / (sigma_hat * np.sqrt(tau)),
              0)
bs_prob = stats.norm.cdf(d2)

# Add to features
features['xgb_prob'] = xgb_prob
features['xgb_uncertainty'] = xgb_uncertainty
features['bs_implied_prob'] = bs_prob
feat_cols += ['xgb_prob', 'xgb_uncertainty', 'bs_implied_prob']

xgb_val_acc = ((xgb_prob[split_idx:] > 0.5) == y_val_xgb).mean()
print(f'XGBoost val accuracy: {xgb_val_acc:.4f}')
print(f'Total input dimensions: {len(feat_cols)}')

## Dataset & DataLoader

In [ ]:
class SequenceDataset(Dataset):
    """Sliding window sequences for LSTM."""
    def __init__(self, X, y, seq_len=SEQUENCE_LENGTH):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        return self.X[idx:idx + self.seq_len], self.y[idx + self.seq_len]


# Prepare data with purge gap
X_all = features[feat_cols].fillna(0).values
y_all = features[target_col].values

# Train / Val / Test split (80/10/10) with purge
n = len(X_all)
train_end = int(n * 0.8)
val_end = int(n * 0.9)
purge = PURGE_MINUTES  # gap between sets

train_ds = SequenceDataset(X_all[:train_end], y_all[:train_end])
val_ds = SequenceDataset(X_all[train_end + purge:val_end], y_all[train_end + purge:val_end])
test_ds = SequenceDataset(X_all[val_end + purge:], y_all[val_end + purge:])

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=512)
test_dl = DataLoader(test_ds, batch_size=512)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

## LSTM-MDN Model

In [ ]:
class LSTM_MDN(nn.Module):
    """LSTM → Student-t Mixture Density Network."""
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, n_layers=N_LSTM_LAYERS,
                 k=K_COMPONENTS, dropout=DROPOUT):
        super().__init__()
        self.layer_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers,
                           batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, k * 4)  # pi, mu, sigma, nu per component
        )
        self.k = k

    def forward(self, x):
        x = self.layer_norm(x)
        out, _ = self.lstm(x)
        h = out[:, -1, :]  # last timestep
        params = self.fc(h)

        # Split into K components × 4 parameters
        pi, mu, sigma, nu = params.chunk(4, dim=-1)
        pi = F.softmax(pi, dim=-1)          # mixture weights sum to 1
        sigma = F.softplus(sigma) + 1e-6     # positive scale
        nu = F.softplus(nu) + 2.1            # df > 2 for finite variance
        return pi, mu, sigma, nu

In [ ]:
def student_t_log_prob(x, mu, sigma, nu):
    """Log PDF of Student-t distribution."""
    z = (x - mu) / sigma
    log_norm = (torch.lgamma((nu + 1) / 2) - torch.lgamma(nu / 2)
                - 0.5 * torch.log(nu * np.pi) - torch.log(sigma))
    log_body = -(nu + 1) / 2 * torch.log(1 + z**2 / nu)
    return log_norm + log_body


def mixture_nll_loss(pi, mu, sigma, nu, target):
    """Negative log-likelihood of Student-t mixture."""
    target = target.unsqueeze(-1).expand_as(mu)
    log_probs = torch.log(pi + 1e-8) + student_t_log_prob(target, mu, sigma, nu)
    return -torch.logsumexp(log_probs, dim=-1).mean()


def focal_loss(pred_prob, target, gamma_low=5, gamma_high=3):
    """FLSD-53 focal loss for calibration."""
    pred_prob = pred_prob.clamp(1e-6, 1 - 1e-6)
    gamma = torch.where(pred_prob < 0.2, gamma_low, gamma_high)
    bce = F.binary_cross_entropy(pred_prob, target, reduction='none')
    weight = (1 - torch.exp(-bce)) ** gamma
    return (weight * bce).mean()


def model_prob_up(pi, mu, sigma, nu):
    """P(return > 0) from mixture — differentiable approximation."""
    # Use normal CDF approximation for gradient flow
    z = -mu / sigma  # standardized value at strike=0
    # Approximate Student-t CDF with normal for training stability
    cdf_approx = 0.5 * (1 + torch.erf(z / np.sqrt(2)))
    p_up = (pi * (1 - cdf_approx)).sum(dim=-1)
    return p_up

## Training

In [ ]:
def train_one_model(train_dl, val_dl, input_dim, seed=42):
    """Train a single LSTM-MDN model. Returns model and best val loss."""
    torch.manual_seed(seed)
    model = LSTM_MDN(input_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE
    )

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(MAX_EPOCHS):
        # Train
        model.train()
        train_losses = []
        for X_batch, y_batch in train_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pi, mu, sigma, nu = model(X_batch)

            # Target return (use small proxy since we predict binary)
            target_return = (y_batch - 0.5) * 0.01  # scale to return space
            nll = mixture_nll_loss(pi, mu, sigma, nu, target_return)

            p_up = model_prob_up(pi, mu, sigma, nu)
            fl = focal_loss(p_up, y_batch)

            loss = nll + FOCAL_LAMBDA * fl
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        scheduler.step()

        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_dl:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pi, mu, sigma, nu = model(X_batch)
                target_return = (y_batch - 0.5) * 0.01
                val_losses.append(mixture_nll_loss(pi, mu, sigma, nu, target_return).item())

        val_loss = np.mean(val_losses)
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}  train_loss={np.mean(train_losses):.4f}  val_loss={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model, best_val_loss

In [ ]:
# Deep Ensemble: train multiple models with different seeds
input_dim = len(feat_cols)
ensemble = []

for i in range(ENSEMBLE_SIZE):
    print(f'Training model {i+1}/{ENSEMBLE_SIZE}')
    model, val_loss = train_one_model(train_dl, val_dl, input_dim, seed=42 + i)
    ensemble.append(model)
    torch.save(model.state_dict(), MODEL_DIR / f'lstm_mdn_{i}.pt')
    print(f'  Best val loss: {val_loss:.4f}\n')

print(f'Ensemble of {len(ensemble)} models saved')

## Inference & Calibration

In [ ]:
def ensemble_predict(models, dataloader):
    """Average predictions across ensemble members."""
    all_probs = []
    all_targets = []

    for model in models:
        model.eval()
        probs = []
        targets = []
        with torch.no_grad():
            for X_batch, y_batch in dataloader:
                X_batch = X_batch.to(device)
                pi, mu, sigma, nu = model(X_batch)

                # Compute P(up) using scipy for exact Student-t CDF
                pi_np = pi.cpu().numpy()
                mu_np = mu.cpu().numpy()
                sigma_np = sigma.cpu().numpy()
                nu_np = nu.cpu().numpy()

                batch_probs = []
                for j in range(len(pi_np)):
                    p = compute_strike_probability(
                        pi_np[j], mu_np[j], sigma_np[j], nu_np[j], strike_return=0.0
                    )
                    batch_probs.append(p)
                probs.extend(batch_probs)
                targets.extend(y_batch.numpy())

        all_probs.append(probs)
        all_targets = targets  # same for all models

    # Average across ensemble
    avg_probs = np.mean(all_probs, axis=0)
    return np.array(avg_probs), np.array(all_targets)


test_probs, test_targets = ensemble_predict(ensemble, test_dl)
print(f'Test predictions: {len(test_probs)}')

In [ ]:
# Venn-ABERS calibration (simplified: isotonic regression)
# Use last portion of validation set as calibration set
val_probs, val_targets = ensemble_predict(ensemble, val_dl)

# Fit isotonic regression
iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(val_probs, val_targets)

# Calibrate test predictions
calibrated_probs = iso_reg.predict(test_probs)
print('Calibration applied')

## CPCV Validation

In [ ]:
def cpcv_evaluate(X, y, n_groups=10, k_test=2):
    """Combinatorial Purged Cross-Validation.
    Returns Brier scores across all C(n,k) paths.
    """
    from itertools import combinations
    n = len(X)
    group_size = n // n_groups
    groups = [list(range(i * group_size, min((i + 1) * group_size, n)))
              for i in range(n_groups)]

    brier_scores = []
    combos = list(combinations(range(n_groups), k_test))

    for test_groups in combos:
        test_idx = []
        for g in test_groups:
            test_idx.extend(groups[g])
        train_idx = [i for i in range(n) if i not in test_idx]

        # Purge: remove PURGE_MINUTES around test boundaries
        test_set = set(test_idx)
        purge_idx = set()
        for idx in test_idx:
            for p in range(1, PURGE_MINUTES + 1):
                purge_idx.add(idx - p)
                purge_idx.add(idx + p)
        train_idx = [i for i in train_idx if i not in purge_idx and 0 <= i < n]

        if len(train_idx) < 100 or len(test_idx) < 10:
            continue

        # Quick XGBoost for CPCV (faster than retraining LSTM per fold)
        clf = xgb.XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            random_state=42, eval_metric='logloss'
        )
        clf.fit(X[train_idx], y[train_idx], verbose=False)
        pred = clf.predict_proba(X[test_idx])[:, 1]
        brier = brier_score_loss(y[test_idx], pred)
        brier_scores.append(brier)

    return np.array(brier_scores)


cpcv_briers = cpcv_evaluate(X_all, y_all)
print(f'CPCV Brier scores ({len(cpcv_briers)} paths):')
print(f'  Mean: {cpcv_briers.mean():.4f}  Std: {cpcv_briers.std():.4f}')
print(f'  Target: < {TARGET_BRIER}')

## Metrics

In [ ]:
# Key metrics on test set
brier = brier_score_loss(test_targets, calibrated_probs)
accuracy = ((calibrated_probs > 0.5) == test_targets).mean()
auc = roc_auc_score(test_targets, calibrated_probs)

print(f'Brier score:  {brier:.4f}  (target < {TARGET_BRIER})')
print(f'Accuracy:     {accuracy:.4f}  (target > {TARGET_ACCURACY})')
print(f'AUC:          {auc:.4f}')

In [ ]:
# Calibration curve (reliability diagram)
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_means = []
bin_freqs = []

for i in range(n_bins):
    mask = (calibrated_probs >= bin_edges[i]) & (calibrated_probs < bin_edges[i + 1])
    if mask.sum() > 0:
        bin_means.append(calibrated_probs[mask].mean())
        bin_freqs.append(test_targets[mask].mean())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
ax.scatter(bin_means, bin_freqs, s=40, zorder=3, label='Model')
ax.set_xlabel('Predicted probability')
ax.set_ylabel('Observed frequency')
ax.set_title('Calibration Curve')
ax.legend()
plt.tight_layout()
plt.show()